# 02 — Modeling Pipeline

**Objectif :** Entraîner et évaluer un modèle LightGBM de prédiction de l'indice de richesse (`wealth_index`), avec validation croisée spatiale et estimation conservative de l'incertitude.

---

## Entrées

| Fichier | Description |
|---------|-------------|
| `data/processed/dhs_prepared_with_buffers.parquet` | Grappes DHS + buffers (Notebook 01) |
| `data/processed/features/cluster_features_gee.parquet` | Features GEE (extraction GEE, mode `clusters`) |

## Jeu de features (v1 / v2)

| Version | Colonne bâti | Source |
|---------|--------------|--------|
| **v1** | `built_density` | WorldCover 2021 |
| **v2** | `ghsl_built_fraction` | GHSL GHS-BUILT-S 2015 |

> **Exécuter les cellules dans l'ordre** (menu *Run All*) pour éviter les erreurs de variables non définies.

In [1]:
import json
import sys
from datetime import datetime, timezone
from pathlib import Path

import lightgbm
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.data.load_training_data import build_training_matrix, load_prepared_clusters
from src.data.merge_features import merge_dhs_with_features
from src.features.load_features import load_cluster_features, resolve_feature_source
from src.models.cv_pipeline import run_spatial_cv
from src.models.evaluate import build_cv_report, compute_metrics, compute_spearman, evaluate_by_strata
from src.models.save_load import save_model
from src.models.train import extract_median_best_iteration, train_final_model
from src.models.uncertainty import attach_prediction_intervals, compute_residual_uncertainty
from src.utils.config import load_config
from src.utils.helpers import hash_config_file
from src.utils.spatial_cv import select_cv_strategy

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
print("✓ Imports réussis")


✓ Imports réussis


## 1. Configuration

In [2]:
# ── Paramètres modifiables ──────────────────────────────────────────────
FEATURE_SET = "v3"          # "v1" | "v2" (GHSL) | "v3" (GHSL + CHIRPS)
USE_FAKE_FEATURES = False   # True = features simulées (test pipeline)

FEATURE_COLUMNS_BY_SET = {
    "v1": [
        "night_lights_mean", "ndvi_mean", "ndbi_mean",
        "dist_road_km", "dist_school_km", "dist_health_km",
        "pop_density", "elevation_m", "slope_deg", "built_density",
    ],
    "v2": [
        "night_lights_mean", "ndvi_mean", "ndbi_mean",
        "dist_road_km", "dist_school_km", "dist_health_km",
        "pop_density", "elevation_m", "slope_deg", "ghsl_built_fraction",
    ],
    "v3": [
        "night_lights_mean", "ndvi_mean", "ndbi_mean",
        "dist_road_km", "dist_school_km", "dist_health_km",
        "pop_density", "elevation_m", "slope_deg", "ghsl_built_fraction",
        "precip_annual_mm", "precip_wet_season_mm", "precip_cv",
    ],
}

# ── Chargement config projet ────────────────────────────────────────────
config_path = PROJECT_ROOT / "configs" / "default.yaml"
config = load_config(config_path)
config_hash = hash_config_file(config_path)

# Surcharge session (notebook > default.yaml)
config["features"]["feature_set"] = FEATURE_SET
config["features"]["columns"] = FEATURE_COLUMNS_BY_SET[FEATURE_SET]
config["features"]["fake"] = USE_FAKE_FEATURES
config["features"]["source"] = "fake" if USE_FAKE_FEATURES else "gee"
config["features"]["gee_parquet"] = "data/processed/features/cluster_features_gee_real.parquet"

feature_source = resolve_feature_source(config)
FEATURE_COLS = config["features"]["columns"]
gee_parquet_path = PROJECT_ROOT / config["features"].get(
    "gee_parquet", "data/processed/features/cluster_features_gee.parquet"
)

print(f"LightGBM version : {lightgbm.__version__}")
print(f"Config hash      : {config_hash}")
print(f"Feature set      : {FEATURE_SET}")
print(f"Feature source   : {feature_source}")
print(f"N features       : {len(FEATURE_COLS)}")
print(f"Dernière colonne : {FEATURE_COLS[-1]}")
if feature_source == "gee":
    print(f"GEE parquet      : {gee_parquet_path}")
    print(f"Parquet existe   : {gee_parquet_path.exists()}")


LightGBM version : 4.6.0
Config hash      : fc8f8c4d93e0ae57b8336d4e4c1a2207
Feature set      : v3
Feature source   : gee
N features       : 13
Dernière colonne : precip_cv
GEE parquet      : C:\~\Projects\cameroon-poverty-mapping\data\processed\features\cluster_features_gee_real.parquet
Parquet existe   : True


## 2. Chargement des grappes DHS

In [3]:
prepared_path = PROJECT_ROOT / config["data"]["prepared_clusters"]
gdf = load_prepared_clusters(prepared_path)

assert gdf["wealth_index"].isna().sum() == 0
assert set(gdf["urban_rural"].unique()).issubset({"urban", "rural"})
assert gdf.crs.to_epsg() == 4326

print(f"Nombre de grappes : {len(gdf)}")
print(f"Régions           : {gdf['region'].nunique()}")
gdf.head()


Nombre de grappes : 430
Régions           : 12


,cluster_id,latitude,longitude,urban_rural,region,wealth_index,buffer_km,geometry,displacement_source
0,1,10.748742,14.607247,urban,Extrême-Nord,-47642.964286,2.0,"POLYGON ((14.62554 10.74876, 14.62545 10.74699...",dhs_official_ge
1,2,3.896324,13.282114,rural,Est,-74227.857143,5.0,"POLYGON ((13.32713 3.89641, 13.32692 3.89198, ...",dhs_official_ge
2,3,4.554889,11.379685,urban,Centre,-9320.035714,2.0,"POLYGON ((11.39768 4.55498, 11.3976 4.55321, 1...",dhs_official_ge
3,5,3.860075,11.547974,urban,Yaoundé,83604.285714,2.0,"POLYGON ((11.56596 3.86015, 11.56588 3.85838, ...",dhs_official_ge
4,7,6.520462,13.187004,rural,Adamaoua,-123314.291667,5.0,"POLYGON ((13.23221 6.52062, 13.232 6.51619, 13...",dhs_official_ge


In [4]:
title_suffix = " (fictives)" if feature_source == "fake" else ""
fig, ax = plt.subplots()
for label, subset in gdf.groupby("urban_rural"):
    ax.scatter(subset["longitude"], subset["latitude"], label=label, alpha=0.8)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title(f"Grappes DHS{title_suffix}")
ax.legend()
plt.show()


C:\Users\HP\AppData\Local\Temp\ipykernel_7768\2605870115.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Chargement des features + construction de `training_df`

Cette section crée **`training_df`** — table fusionnée grappes + features.
Toutes les cellules suivantes en dépendent : exécutez celle-ci avant la CV.

In [5]:
features_df, feature_source = load_cluster_features(gdf, config, project_root=PROJECT_ROOT)

assert len(FEATURE_COLS) == len(FEATURE_COLUMNS_BY_SET[FEATURE_SET]), (
    f"Attendu {len(FEATURE_COLUMNS_BY_SET[FEATURE_SET])} features, obtenu {len(FEATURE_COLS)}"
)
missing_cols = [c for c in FEATURE_COLS if c not in features_df.columns]
assert not missing_cols, f"Colonnes manquantes dans features : {missing_cols}"

training_df = merge_dhs_with_features(gdf, features_df)
assert len(training_df) == len(gdf), "Perte de grappes après fusion"

if feature_source == "fake":
    print("⚠️  Features simulées — pipeline structurel uniquement")
else:
    print("✅ Features GEE chargées depuis :", gee_parquet_path)
    print(f"   Feature set {FEATURE_SET} — colonne bâti : {FEATURE_COLS[-1]}")

print(f"training_df : {training_df.shape[0]} lignes × {training_df.shape[1]} colonnes")
features_df[FEATURE_COLS].describe().round(3)


✅ Features GEE chargées depuis : C:\~\Projects\cameroon-poverty-mapping\data\processed\features\cluster_features_gee_real.parquet
   Feature set v3 — colonne bâti : precip_cv
training_df : 430 lignes × 21 colonnes


,night_lights_mean,ndvi_mean,ndbi_mean,dist_road_km,dist_school_km,dist_health_km,pop_density,elevation_m,slope_deg,ghsl_built_fraction,precip_annual_mm,precip_wet_season_mm,precip_cv
count,430.000,430.000,430.000,430.000,430.000,430.000,430.000,430.000,430.000,430.000,430.000,430.000,430.000
mean,4.389,0.423,-0.012,1.230,7.197,15.848,22.714,642.948,1.283,0.104,1793.326,1707.918,1.854
std,7.866,0.180,0.135,1.699,10.745,17.782,46.682,452.960,1.344,0.140,691.248,648.102,0.454
min,0.141,0.000,-0.306,0.000,0.000,0.042,0.038,5.234,0.023,0.000,459.873,459.872,0.800
25%,0.216,0.268,-0.118,0.421,0.344,1.200,0.731,291.738,0.439,0.000,1444.182,1377.154,1.522
50%,0.388,0.392,0.029,0.834,2.210,7.049,2.269,653.339,0.747,0.015,1663.106,1551.541,1.626
75%,3.411,0.562,0.105,1.609,8.981,29.319,12.273,848.356,1.619,0.184,2121.472,2026.114,2.311
max,39.702,0.827,0.174,22.113,50.000,50.000,197.843,2165.768,9.059,0.499,3463.554,3281.837,2.982


In [6]:
corr_with_target = training_df[FEATURE_COLS].corrwith(training_df["wealth_index"]).abs()
print("Corrélations absolues features ↔ wealth_index :")
print(corr_with_target.sort_values().round(3))

if feature_source == "fake":
    assert corr_with_target.max() < 0.3, "Features fictives trop corrélées à la cible"
else:
    print("Mode GEE : signal réel possible — pas de seuil de corrélation imposé.")


Corrélations absolues features ↔ wealth_index :
precip_cv               0.078
ndbi_mean               0.107
slope_deg               0.129
elevation_m             0.180
ndvi_mean               0.286
dist_road_km            0.506
precip_wet_season_mm    0.515
precip_annual_mm        0.537
dist_school_km          0.549
pop_density             0.552
dist_health_km          0.601
night_lights_mean       0.602
ghsl_built_fraction     0.741
dtype: float64
Mode GEE : signal réel possible — pas de seuil de corrélation imposé.


In [7]:
X, y, meta = build_training_matrix(training_df, feature_cols=FEATURE_COLS)
print(f"X shape : {X.shape}")
print(f"y shape : {y.shape}")


X shape : (430, 13)
y shape : (430,)


In [8]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(y, kde=True, ax=axes[0])
axes[0].set_title("Distribution de wealth_index")
sns.boxplot(data=training_df, x="urban_rural", y="wealth_index", ax=axes[1])
axes[1].set_title("wealth_index par urbain/rural")
plt.tight_layout()
plt.show()


C:\Users\HP\AppData\Local\Temp\ipykernel_7768\2002112515.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Validation croisée spatiale

Validation par **blocs spatiaux** pour éviter le *spatial leakage*.
Repli automatique vers `region_based_cv` si les plis sont déséquilibrés.

In [9]:
cv_strategy, fold_ids, balance_report = select_cv_strategy(
    gdf,
    preferred=config["model"]["cv_strategy"],
    n_folds=config["model"]["n_folds"],
    random_state=config["model"]["random_state"],
)
print(f"Stratégie retenue : {cv_strategy}")
print(json.dumps(balance_report, indent=2, ensure_ascii=False))


Stratégie retenue : block
{
  "is_balanced": true,
  "folds": [
    {
      "fold": 0,
      "n_val": 209,
      "n_train": 221,
      "urban_rural_counts": {
        "urban": 127,
        "rural": 82
      }
    },
    {
      "fold": 1,
      "n_val": 102,
      "n_train": 328,
      "urban_rural_counts": {
        "urban": 72,
        "rural": 30
      }
    },
    {
      "fold": 2,
      "n_val": 27,
      "n_train": 403,
      "urban_rural_counts": {
        "rural": 19,
        "urban": 8
      }
    },
    {
      "fold": 3,
      "n_val": 28,
      "n_train": 402,
      "urban_rural_counts": {
        "rural": 18,
        "urban": 10
      }
    },
    {
      "fold": 4,
      "n_val": 64,
      "n_train": 366,
      "urban_rural_counts": {
        "rural": 44,
        "urban": 20
      }
    }
  ]
}


In [10]:
# Garde-fou : training_df doit exister (section 3)
if "training_df" not in globals():
    raise NameError(
        "training_df n'est pas défini. Exécutez la section 3 "
        "(Chargement des features) avant cette cellule."
    )

training_df = training_df.copy()
training_df["fold_id"] = fold_ids.values

fig, ax = plt.subplots()
scatter = ax.scatter(
    training_df["longitude"],
    training_df["latitude"],
    c=training_df["fold_id"],
    cmap="tab10",
    s=80,
)
ax.set_title(f"Grappes par pli CV ({cv_strategy})")
plt.colorbar(scatter, ax=ax, label="fold_id")
plt.show()


C:\Users\HP\AppData\Local\Temp\ipykernel_7768\2471188551.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Entraînement CV + métriques OOF

In [11]:
cv_results = run_spatial_cv(
    X, y, gdf, config, cv_strategy=cv_strategy, return_models=True
)

fold_metrics_df = pd.DataFrame(cv_results.fold_metrics)
fold_metrics_df


,r2,rmse,mae,spearman,fold,n_train,n_val,best_iteration
0,0.637170,43281.389661,33471.185970,0.801815,0,221,209,68
1,0.798798,36930.160082,26884.299512,0.820389,1,328,102,81
2,0.646600,30548.097203,25706.947310,0.608669,2,403,27,58
3,0.563907,42935.140101,32762.358219,0.724138,3,402,28,80
4,0.568961,35011.407209,28669.001531,0.615751,4,366,64,132


In [12]:
metrics_global = compute_metrics(y, cv_results.oof_predictions)
metrics_global["spearman"] = compute_spearman(y, cv_results.oof_predictions)
pd.DataFrame([metrics_global])


,r2,rmse,mae,spearman
0,0.77564,39938.606041,30660.29331,0.874525


In [13]:
fig, ax = plt.subplots()
ax.scatter(y, cv_results.oof_predictions, alpha=0.8)
lims = [min(y.min(), cv_results.oof_predictions.min()), max(y.max(), cv_results.oof_predictions.max())]
ax.plot(lims, lims, "r--", label="Identité")
ax.set_xlabel("wealth_index observé")
ax.set_ylabel("wealth_index prédit (OOF)")
ax.set_title(f"Prédictions OOF — R²={metrics_global['r2']:.3f}")
ax.legend()
plt.show()


C:\Users\HP\AppData\Local\Temp\ipykernel_7768\2400923907.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [14]:
metrics_urban_rural = evaluate_by_strata(y, cv_results.oof_predictions, meta["urban_rural"])
metrics_region = evaluate_by_strata(y, cv_results.oof_predictions, meta["region"])
print("Par urbain/rural")
display(metrics_urban_rural)
print("Par région")
display(metrics_region)


Par urbain/rural


,stratum,n,r2,rmse,mae,spearman
0,rural,193,0.274631,36509.880261,27670.332571,0.649775
1,urban,237,0.479239,42527.042611,33095.155854,0.668861


Par région


,stratum,n,r2,rmse,mae,spearman
0,Adamaoua,34,0.746052,31790.039519,24968.826109,0.801986
1,Centre,39,0.720084,30951.325340,23535.373047,0.810526
2,Douala,44,-0.469880,46854.143524,35720.295761,-0.163377
3,Est,34,0.728098,36465.372726,29766.000888,0.761039
4,Extrême-Nord,48,0.498455,38182.473888,32004.383200,0.512484
5,Littoral,32,0.327430,43578.235457,32337.755591,0.710411
6,Nord,40,0.701589,34345.553528,25611.800941,0.742402
7,Nord-Ouest,28,0.655342,40926.216012,30242.163059,0.833607
8,Ouest,42,0.538959,36036.841278,28052.714162,0.788510
9,Sud,32,0.482395,44937.556401,37514.912117,0.711144


In [15]:
best_fold_idx = int(fold_metrics_df["rmse"].idxmin())
importance_df = cv_results.models[best_fold_idx].feature_importance()
importance_df.plot.bar(x="feature", y="gain", legend=False, title="Importance (gain LightGBM)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()
importance_df


C:\Users\HP\AppData\Local\Temp\ipykernel_7768\2772436535.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,feature,gain
0,night_lights_mean,1.951614e+13
1,precip_annual_mm,1.429797e+12
2,pop_density,6.214802e+11
3,dist_road_km,5.709783e+11
4,ghsl_built_fraction,4.549951e+11
5,ndvi_mean,3.453596e+11
6,elevation_m,2.710891e+11
7,dist_health_km,2.343661e+11
8,dist_school_km,2.117558e+11
9,precip_cv,1.944595e+11


## 6. Incertitude (approximation conservative)

Intervalles à 90 % basés sur les résidus OOF globaux — pas une incertitude pixel.

In [16]:
uncertainty = compute_residual_uncertainty(cv_results.oof_residuals)
oof_df = attach_prediction_intervals(
    cv_results.oof_predictions,
    y,
    uncertainty,
    meta=meta,
)
print(json.dumps(uncertainty, indent=2, ensure_ascii=False))
oof_df.head()


{
  "confidence_level": 0.9,
  "residual_std": 39871.44685049113,
  "residual_q90": 66802.52942869898,
  "interval_half_width": 66802.52942869898,
  "method": "global_residual_quantile_oof",
  "note": "Approximation conservative basée sur les résidus OOF globaux. Ne pas utiliser pour décisions opérationnelles."
}


,y_true,y_oof_pred,residual,lower_90,upper_90,cluster_id,region,urban_rural
0,-47642.964286,-26223.632025,-21419.332261,-93026.161454,40578.897404,1,Extrême-Nord,urban
1,-74227.857143,-67302.498779,-6925.358364,-134105.028208,-499.969350,2,Est,rural
2,-9320.035714,-29555.159009,20235.123295,-96357.688438,37247.370420,3,Centre,urban
3,83604.285714,92104.829513,-8500.543798,25302.300084,158907.358941,5,Yaoundé,urban
4,-123314.291667,-82991.419452,-40322.872215,-149793.948881,-16188.890023,7,Adamaoua,rural


In [17]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(cv_results.oof_residuals, kde=True, ax=axes[0])
axes[0].set_title("Résidus OOF")
fold_metrics_df.plot.bar(x="fold", y="rmse", ax=axes[1], legend=False, title="RMSE par pli")
plt.tight_layout()
plt.show()


C:\Users\HP\AppData\Local\Temp\ipykernel_7768\36687974.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Modèle final + export

In [18]:
median_iter = extract_median_best_iteration(cv_results)
final_model = train_final_model(
    X,
    y,
    config,
    median_best_iteration=median_iter,
    strata=meta["urban_rural"],
)

models_dir = PROJECT_ROOT / config["output"]["models_dir"]
models_dir.mkdir(parents=True, exist_ok=True)
model_suffix = "fake" if feature_source == "fake" else f"gee_{FEATURE_SET}"
model_path = models_dir / f"wealth_model_lgbm_v0_{model_suffix}.pkl"
save_model(final_model, model_path)

print("⚠️  Les métriques de référence sont uniquement celles de la CV OOF.")
print(f"Modèle final sauvegardé : {model_path}")
print(f"Median best iteration  : {median_iter}")


⚠️  Les métriques de référence sont uniquement celles de la CV OOF.
Modèle final sauvegardé : C:\~\Projects\cameroon-poverty-mapping\models\wealth_model_lgbm_v0_gee_v3.pkl
Median best iteration  : 80


In [19]:
reports_dir = PROJECT_ROOT / config["output"]["reports_dir"]
training_dir = PROJECT_ROOT / config["data"]["training_dir"]
logs_dir = PROJECT_ROOT / config["output"]["logs_dir"]
for d in [reports_dir, training_dir, logs_dir]:
    d.mkdir(parents=True, exist_ok=True)

report = build_cv_report(
    cv_results.fold_metrics,
    metrics_global,
    config_hash=config_hash,
    cv_strategy=cv_results.cv_strategy,
    fake_data=(feature_source == "fake"),
)
report["feature_source"] = feature_source
report["feature_set"] = FEATURE_SET
report_path = reports_dir / "cv_metrics.json"
report_path.write_text(json.dumps(report, indent=2), encoding="utf-8")

oof_path = training_dir / "oof_predictions.parquet"
oof_df.to_parquet(oof_path, index=False)

training_df[FEATURE_COLS + ["cluster_id", "wealth_index", "region", "urban_rural", "fold_id"]].to_parquet(
    training_dir / "training_matrix.parquet", index=False
)

importance_df.to_csv(reports_dir / "feature_importance_gain.csv", index=False)

log_payload = {
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "config_hash": config_hash,
    "cv_strategy": cv_results.cv_strategy,
    "feature_source": feature_source,
    "feature_set": FEATURE_SET,
    "fake_data": feature_source == "fake",
    "n_clusters": len(gdf),
    "metrics_oof": metrics_global,
}
(logs_dir / f"run_{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')}.json").write_text(
    json.dumps(log_payload, indent=2), encoding="utf-8"
)

print(f"Rapport CV : {report_path}")
print(f"OOF preds  : {oof_path}")


Rapport CV : C:\~\Projects\cameroon-poverty-mapping\outputs\reports\cv_metrics.json
OOF preds  : C:\~\Projects\cameroon-poverty-mapping\data\processed\training\oof_predictions.parquet


In [20]:
print("=== Limites selon le mode actuel ===")
print(f"Mode features : {feature_source}")
print(f"Feature set   : {FEATURE_SET} ({FEATURE_COLS[-1]})")

if feature_source == "fake":
    print("- Grappes et wealth_index simulés")
    print("- Features décorrélées de la cible (pipeline structurel)")
    print("- Métriques non interprétables scientifiquement")
else:
    print("- Labels DHS soumis au jitter GPS")
    if FEATURE_SET == "v2":
        print("- Fraction bâtie : GHSL 2015 (ghsl_built_fraction)")
    else:
        print("- Fraction bâtie : WorldCover 2021 (built_density)")
    print("- Métriques exploratoires — pas de décision opérationnelle directe")

print("- Incertitude OOF : approximation conservative (résidus globaux)")
print("- Ne pas utiliser pour ciblage individuel ou allocation budgétaire")


=== Limites selon le mode actuel ===
Mode features : gee
Feature set   : v3 (precip_cv)
- Labels DHS soumis au jitter GPS
- Fraction bâtie : WorldCover 2021 (built_density)
- Métriques exploratoires — pas de décision opérationnelle directe
- Incertitude OOF : approximation conservative (résidus globaux)
- Ne pas utiliser pour ciblage individuel ou allocation budgétaire


## 8. Comparaison v1 vs v2 (WorldCover vs GHSL)

Exécute les deux jeux de features sur les **mêmes grappes** et compare les métriques OOF.

> Automatisation : `python scripts/compare_v1_v2_ghsl.py`

In [21]:
# Comparaison objective v1 (built_density) vs v2 (ghsl_built_fraction)
# Nécessite : cluster_features_gee_v1.parquet + cluster_features_gee.parquet (v2)

import subprocess

compare_script = PROJECT_ROOT / "scripts" / "compare_v1_v2_ghsl.py"
result = subprocess.run(
    [sys.executable, str(compare_script), "--skip-extraction"],
    cwd=str(PROJECT_ROOT),
    capture_output=True,
    text=True,
)
print(result.stdout)
if result.returncode != 0:
    print("stderr:", result.stderr)
    print("ℹ️  Si v1 parquet manquant, relancez sans --skip-extraction :")
    print(f"   python {compare_script}")

comparison_path = PROJECT_ROOT / "outputs/reports/v1_vs_v2_comparison.json"
if comparison_path.exists():
    comparison = json.loads(comparison_path.read_text(encoding="utf-8"))
    table = comparison["comparison_table"]
    comp_df = pd.DataFrame(table)
    display(comp_df)
    print("\nConclusion :", comparison["conclusion"])
    print("Verdict    :", comparison["verdict"])


ℹ️  Parquet v1 existant : C:\~\Projects\cameroon-poverty-mapping\data\processed\features\cluster_features_gee_v1.parquet

📊 Pipeline v1 (WorldCover)...

stderr: Traceback (most recent call last):
  File "C:\~\Projects\cameroon-poverty-mapping\scripts\compare_v1_v2_ghsl.py", line 241, in <module>
    raise SystemExit(main())
                     ~~~~^^
  File "C:\~\Projects\cameroon-poverty-mapping\scripts\compare_v1_v2_ghsl.py", line 219, in main
    v1_result = run_pipeline(
        feature_set="v1",
        gee_parquet=V1_PARQUET,
        save_artifacts=False,
    )
  File "C:\~\Projects\cameroon-poverty-mapping\scripts\run_notebook_02_pipeline.py", line 220, in run_pipeline
    features_df, loaded_source = load_cluster_features(gdf, config, project_root=PROJECT_ROOT)
                                 ~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\~\Projects\cameroon-poverty-mapping\src\features\load_features.py", line 76, in load_cluster_features
    raise V

,metric,v1_worldcover,v2_ghsl,delta_v2_minus_v1
0,R² OOF,-0.3387,-0.3387,0.0
1,Spearman OOF,-0.3223,-0.3223,0.0
2,RMSE OOF,1.0694,1.0694,0.0
3,Gain built_density,0.0000,NaN,NaN
4,Gain ghsl_built_fraction,NaN,0.0000,NaN
5,Rang built_density,10.0000,NaN,NaN
6,Rang ghsl_built_fraction,NaN,10.0000,NaN



Conclusion : Sur grappes fictives, v1 et v2 sont équivalents (écart métrique < 2 pts). GHSL reste préférable pour l'alignement temporel DHS 2018 ; valider sur vraies DHS.
Verdict    : equivalent_on_fake_data


## Prochaines étapes

1. Valider v1 vs v2 sur **vraies grappes DHS 2018**
2. Export raster national + cartes (Notebook 04)
3. Interprétabilité SHAP